In [4]:
import json
import os
from tqdm import tqdm
from dotenv import load_dotenv
import hashlib
import spacy

from elasticsearch import Elasticsearch

load_dotenv()
ES_HOST = os.getenv("ES_HOST", "http://localhost:9200")
INDEX_NAME = os.getenv("INDEX_NAME")

es_client = Elasticsearch('http://localhost:9200')

In [5]:
with open("data/putin_complete.json", "r") as f:
    speeches = json.load(f)

try:
    es_client.indices.delete(index=INDEX_NAME)
except:
    print(f"Index {INDEX_NAME} already exists")

mapping = {
    "properties": {
        "id":   {"type": "integer"},
        "unique_hash": {"type": "text"},
        "title": {"type": "text"},
        "text":  {"type": "text"},
        "date": {"type": "date"},
    }
}

es_client.indices.create(
    index=INDEX_NAME,
    mappings=mapping,
)

index = 0

for doc in tqdm(speeches, "Indexing..."):
    subset_speech = {k: doc[k] for k in doc.keys() if k in ["date", "title", "transcript_filtered"]}
    subset_speech["text"] = subset_speech["transcript_filtered"]
    subset_speech.pop("transcript_filtered")
    subset_speech["unique_hash"] = hashlib.sha256(subset_speech["text"].encode()).hexdigest()[:8]
    index += 1
    es_client.index(index=INDEX_NAME, id=index, document=subset_speech)


descriptions_path = "../" + os.getenv("DATASET_DESCRIPTIONS")

with open(descriptions_path, "r") as f:
    descriptions = json.load(f)

descriptions[INDEX_NAME] = [f"Base dataset: {INDEX_NAME}"]

with open(descriptions_path, 'w') as file:
    json.dump(descriptions, file)

Index russian_speeches already exists


Indexing...: 100%|██████████| 9838/9838 [01:04<00:00, 152.96it/s]
